# 03 - The Bias-Variance Tradeoff

In this notebook, we visualize Underfitting (High Bias) and Overfitting (High Variance) by fitting polynomial curves to a sine wave.

## Visual Demonstration
We will generate a noisy sine wave and try to fit it with lines of varying complexity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

# Generate a noisy sine wave
np.random.seed(0)
n_samples = 30
X = np.sort(np.random.rand(n_samples)) * 2 * np.pi
y = np.sin(X) + np.random.randn(n_samples) * 0.2
X = X[:, np.newaxis]

# X values for plotting smooth curves
X_plot = np.linspace(0, 2*np.pi, 100)[:, np.newaxis]

plt.figure(figsize=(15, 5))

degrees = [1, 3, 15]
titles = ['Underfitting (High Bias)', 'Good Fit (Optimal)', 'Overfitting (High Variance)']

for i, degree in enumerate(degrees):
    ax = plt.subplot(1, 3, i + 1)
    
    # Create a pipeline that creates polynomial features, then runs linear regression
    polynomial_features = PolynomialFeatures(degree=degree, include_bias=False)
    linear_regression = LinearRegression()
    pipeline = Pipeline([('polynomial_features', polynomial_features),
                         ('linear_regression', linear_regression)])
    
    pipeline.fit(X, y)
    y_plot = pipeline.predict(X_plot)
    
    plt.scatter(X, y, color='black', label='Training Data')
    plt.plot(X_plot, y_plot, color='blue', label=f'Degree {degree}')
    plt.title(titles[i])
    plt.ylim((-1.5, 1.5))
    plt.legend(loc='upper right')

plt.tight_layout()
plt.show()

## Interpretation
* **Degree 1 (Linear)**: The model is too simple. It has High Bias. It misses the fundamental curve of the data.
* **Degree 3**: The model captures the underlying signal while ignoring the noise.
* **Degree 15**: The model is too complex. It has High Variance. It connects the dots perfectly but creates a wild, unstable curve that will fail on new data.

## Evaluating Complexity vs Error
Let's plot the actual Training vs Validation error curves.

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)

train_errors = []
val_errors = []
degree_range = range(1, 12)

for d in degree_range:
    model = Pipeline([('poly', PolynomialFeatures(degree=d)),
                      ('lin', LinearRegression())])
    model.fit(X_train, y_train)
    
    train_preds = model.predict(X_train)
    val_preds = model.predict(X_val)
    
    train_errors.append(mean_squared_error(y_train, train_preds))
    val_errors.append(mean_squared_error(y_val, val_preds))

plt.figure(figsize=(8, 5))
plt.plot(degree_range, train_errors, label='Training Error', marker='o')
plt.plot(degree_range, val_errors, label='Validation Error', marker='s')
plt.xlabel('Polynomial Degree (Model Complexity)')
plt.ylabel('Mean Squared Error')
plt.title('Validation Curve (Bias-Variance Tradeoff)')
plt.legend()
plt.ylim(0, 0.5)
plt.show()

Notice how Training Error always goes down as complexity increases, but Validation Error hits a sweet spot (around Degree 3-5) and then explodes upwards. That explosion is Overfitting.

## Industry Notes
In modern Deep Learning, we sometimes observe 'Double Descent', where pushing model complexity to an extreme actually brings the validation error back down. However, for traditional tabular ML, the U-shaped curve holds true.

## Exercises
1. Modify the noise multiplier in the dataset generation from `0.2` to `0.8`. How does this affect the optimal polynomial degree?
2. Add L2 Regularization (Ridge Regression) to the Degree 15 model. Can you smooth out the wild curve without dropping the degree?